In [1]:
#Imports


import shutil
import random
from keras.utils import image_dataset_from_directory
import keras

import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

In [10]:
source_dir = Path(r"C:\Users\simaj\OneDrive - NOVAIMS\2 semestre\Deep Learning\Project\wikiart\wikiart")


In [11]:
img_exts = {".jpg", ".jpeg", ".png"}

In [12]:
class_dirs = sorted(source_dir.iterdir())
class_names = [d.name for d in class_dirs]
n_classes = len(class_names)

### basic information

In [13]:
print(f"Number of authors : {n_classes}")
print(f"Authors           : {class_names}\n")

Number of authors : 23
Authors           : ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']



In [15]:

counts = {}
for class_dir in class_dirs:
    images = [f for f in class_dir.iterdir() if f.suffix.lower() in img_exts]
    counts[class_dir.name] = len(images)

total = sum(counts.values())
print(f"Total paintings    : {total}")
print(f"Min : {min(counts.values())} ({min(counts, key=counts.get)})")
print(f"Max : {max(counts.values())} ({max(counts, key=counts.get)})\n")

Total paintings    : 13340
Min : 336 (Salvador_Dali)
Max : 1322 (Vincent_van_Gogh)



In [17]:
print("Paintings per author:")
for name, count in sorted(counts.items(), key=lambda x: -x[1]):

    print(f" {name:30s} : {count:4d}")

Paintings per author:
 Vincent_van_Gogh               : 1322
 Nicholas_Roerich               : 1274
 Pierre_Auguste_Renoir          :  975
 Claude_Monet                   :  934
 Pyotr_Konchalovsky             :  644
 Camille_Pissarro               :  621
 Albrecht_Durer                 :  580
 John_Singer_Sargent            :  549
 Rembrandt                      :  544
 Marc_Chagall                   :  536
 Pablo_Picasso                  :  534
 Gustave_Dore                   :  528
 Boris_Kustodiev                :  444
 Edgar_Degas                    :  428
 Paul_Cezanne                   :  406
 Ivan_Aivazovsky                :  404
 Martiros_Saryan                :  403
 Eugene_Boudin                  :  389
 Childe_Hassam                  :  385
 Ilya_Repin                     :  378
 Ivan_Shishkin                  :  364
 Raphael_Kirchner               :  362
 Salvador_Dali                  :  336


In [21]:
inbalance_ratio = max(counts.values()) / min(counts.values())
print(f"\ninbalance ratio (max/min) : {inbalance_ratio:.2f}x")
if inbalance_ratio > 3:
    print("Significant inbalance")
else:
    print("NO significant inbalance")


inbalance ratio (max/min) : 3.93x
Significant inbalance



### Data preprocessing

In [22]:
data_dir_path = Path("wikiart_split")

In [23]:
train_ratio = 0.70
val_ratio = 0.15
seed = 123
batch_size = 32
image_size = (224, 224)
input_shape = (224, 224, 3)

In [24]:
# spliting the dataset

if not data_dir_path.exists():
    random.seed(seed)

    for class_dir in sorted(source_dir.iterdir()):
        images = [f for f in class_dir.iterdir() if f.suffix.lower() in img_exts]
        random.shuffle(images)

        n_train = int(len(images) * train_ratio)
        n_val   = int(len(images) * val_ratio)

        splits = {
            "train": images[:n_train],
            "val"  : images[n_train:n_train + n_val],
            "test" : images[n_train + n_val:]
        }

        for split_name, files in splits.items():
            dest = data_dir_path / split_name / class_dir.name
            dest.mkdir(parents=True, exist_ok=True)
            for f in files:
                shutil.copy2(f, dest / f.name)

    print("Dataset split complete.")
else:
    print("Split already exists — skipping.")

Dataset split complete.


In [25]:
# loading the datasets

train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical", 
    batch_size=batch_size,
    image_size=image_size,
    shuffle=True, 
    seed=seed,
    verbose=True
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=batch_size,
    image_size=image_size, 
    shuffle=False, 
    verbose=True
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical", 
    batch_size=batch_size,
    image_size=image_size, 
    shuffle=False, 
    verbose=True
)

class_names = train_ds.class_names
n_classes   = len(class_names)
print(f"\nClasses ({n_classes}): {class_names}")

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.

Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']


In [ ]:
# normalisation 

# bla bla bla

In [ ]:
# augmentation 

# bla bla bla